In [5]:
import numpy as np

# ----------------------------------------------------
# 1. 初始化超参数与网络结构
# ----------------------------------------------------
np.random.seed(42)

N = 200   # Batch Size
D = 2     # 输入特征数 (2维平面点)
H = 64    # 隐藏层神经元个数
C = 3     # 3 分类问题

# 模拟非线性数据：螺旋分布/环形分布数据
X = np.random.randn(N, D)
# 标签 y 为 0, 1, 2
y = np.random.randint(0, C, size=N)

# 参数初始化 (He 初始化 / Kaiming 初始化原则，乘以 sqrt(2/D))
W1 = np.random.randn(D, H) * np.sqrt(2.0 / D)
b1 = np.zeros((1, H))

W2 = np.random.randn(H, C) * np.sqrt(2.0 / H)
b2 = np.zeros((1, C))

learning_rate = 0.1
epochs = 300

# ----------------------------------------------------
# 2. 训练循环
# ----------------------------------------------------
for epoch in range(epochs):
    # ================================================
    # A. 前向传播 (Forward Pass)
    # ================================================
    # 1) 隐藏层 1: Z1 = X @ W1 + b1
    Z1 = np.dot(X, W1) + b1
    
    # 2) ReLU 激活: A1 = max(0, Z1)
    A1 = np.maximum(0, Z1)
    
    # 3) 输出层: Z2 = A1 @ W2 + b2
    Z2 = np.dot(A1, W2) + b2
    
    # 4) Softmax 归一化
    Z2_max = np.max(Z2, axis=1, keepdims=True)
    exp_Z2 = np.exp(Z2 - Z2_max)
    A2 = exp_Z2 / np.sum(exp_Z2, axis=1, keepdims=True)
    
    # 5) 交叉熵 Loss
    correct_log_probs = -np.log(A2[np.arange(N), y] + 1e-15)
    loss = np.mean(correct_log_probs)
    
    # ================================================
    # B. 反向传播 (Backward Pass - 链式法则倒推)
    # ================================================
    # Step 1: 输出层得分梯度 dZ2 = (A2 - Y_onehot) / N
    dZ2 = A2.copy()
    dZ2[np.arange(N), y] -= 1.0
    dZ2 /= N
    
    # Step 2: 算 W2 和 b2 的梯度
    dW2 = np.dot(A1.T, dZ2)                      # (H, N) @ (N, C) -> (H, C)
    db2 = np.sum(dZ2, axis=0, keepdims=True)     # (1, C)
    
    # Step 3: 反向回传到隐藏层激活值 dA1
    dA1 = np.dot(dZ2, W2.T)                      # (N, C) @ (C, H) -> (N, H)
    
    # Step 4: 穿透 ReLU 导数 (Z1 <= 0 的位置梯度清零)
    dZ1 = dA1.copy()
    dZ1[Z1 <= 0] = 0.0                          # (N, H)
    
    # Step 5: 算 W1 和 b1 的梯度
    dW1 = np.dot(X.T, dZ1)                       # (D, N) @ (N, H) -> (D, H)
    db1 = np.sum(dZ1, axis=0, keepdims=True)     # (1, H)
    
    # ================================================
    # C. 参数更新 (SGD)
    # ================================================
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    
    if (epoch + 1) % 50 == 0:
        # 计算当前准确率
        preds = np.argmax(A2, axis=1)
        acc = np.mean(preds == y)
        print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f} | Accuracy: {acc*100:.2f}%")

Epoch  50 | Loss: 1.0594 | Accuracy: 41.00%
Epoch 100 | Loss: 1.0497 | Accuracy: 43.50%
Epoch 150 | Loss: 1.0426 | Accuracy: 44.50%
Epoch 200 | Loss: 1.0366 | Accuracy: 45.00%
Epoch 250 | Loss: 1.0318 | Accuracy: 45.50%
Epoch 300 | Loss: 1.0281 | Accuracy: 46.00%


In [12]:
import math, random

In [22]:
import math
import random

# 1. 矩陣乘法: C = A @ B (A: M x K, B: K x N -> C: M x N)
def matmul(A, B):
    M, K = len(A), len(A[0])
    K2, N = len(B), len(B[0])
    assert K == K2, "矩陣維度不匹配，無法相乘！"
    # 初始化 M x N 的零矩陣
    C = [[0.0 for _ in range(N)] for _ in range(M)]
    for i in range(M):
        for j in range(N):
            for k in range(K):
                C[i][j] += A[i][k] * B[k][j]
    return C

# 2. 矩陣轉置: A^T (M x N -> N x M)
def transpose(A):
    M, N = len(A), len(A[0])
    return [[A[i][j] for i in range(M)] for j in range(N)]

# 3. 矩陣加上偏置向量 (廣播機制: M x N 矩陣每一行加上 1 x N 的向量)
def add_bias(A, b):
    M, N = len(A), len(A[0])
    result = [[0.0 for _ in range(N)] for _ in range(M)]
    for i in range(M):
        for j in range(N):
            result[i][j] = A[i][j] + b[0][j]
    return result

# 4. 矩陣逐元素相減: A - B
def mat_sub(A, B):
    M, N = len(A), len(A[0])
    return [[A[i][j] - B[i][j] for j in range(N)] for i in range(M)]

# 5. 矩陣乘上標量: A * scalar
def mat_scale(A, scalar):
    M, N = len(A), len(A[0])
    return [[A[i][j] * scalar for j in range(N)] for i in range(M)]


In [27]:
def relu_forward(Z):
    M, N = len(Z), len(Z[0])
    A = [[max(0.0, Z[i][j]) for j in range(N)] for i in range(M)]
    return A

def relu_backward(dA, Z):
    M, N = len(Z), len(Z[0])
    dZ = [[0.0 for _ in range(N)] for _ in range(M)]
    for i in range(M):
        for j in range(N):
            # 若前向輸入 Z <= 0，梯度阻斷；否則原樣通過
            dZ[i][j] = dA[i][j] if Z[i][j] > 0 else 0.0
    return dZ


In [28]:
def softmax_cross_entropy_forward(Z, y):
    M, N = len(Z), len(Z[0]) # M 是样本数, N 是类别数
    A = [[0.0 for _ in range(N)] for _ in range(M)]
    total_loss = 0.0

    for i in range(M):
        # 1. 數值穩定性：先求該行的最大值
        max_val = max(Z[i])
        # 2. 求 exp(z - max) 並求和
        exp_row = [math.exp(Z[i][j] - max_val) for j in range(N)]
        sum_exp = sum(exp_row)
        # 3. 歸一化得到概率
        A[i] = [val / sum_exp for val in exp_row]
        # 4. 累加正確類別的 -log(p)
        correct_p = A[i][y[i]]
        total_loss += -math.log(correct_p + 1e-15)

    loss = total_loss / M
    return A, loss

def softmax_cross_entropy_backward(A, y):
    M, N = len(A), len(A[0])
    dZ = [row[:] for row in A] # 深拷貝概率矩陣 A
    
    for i in range(M):
        # 對於正確類別位置，減去 1 (即 A - OneHot)
        dZ[i][y[i]] -= 1.0
        # 除以樣本數求平均
        for j in range(N):
            dZ[i][j] /= M
            
    return dZ


In [29]:
random.seed(42)

# ----------------------------------------------------
# 1. 模擬生成數據 (100 個樣本, 2 維特徵, 3 分類)
# ----------------------------------------------------
N = 100  # Batch Size
D = 2    # 輸入特徵數
H = 6    # 隱藏層個數
C = 3    # 類別數

X = [[random.gauss(0, 1) for _ in range(D)] for _ in range(N)]
y = [random.randint(0, C - 1) for _ in range(N)]

# ----------------------------------------------------
# 2. 初始化網絡參數 (W1, b1, W2, b2)
# ----------------------------------------------------
# W1: D x H (2 x 6), b1: 1 x H (1 x 6)
W1 = [[random.gauss(0, 0.1) for _ in range(H)] for _ in range(D)]
b1 = [[0.0 for _ in range(H)]]

# W2: H x C (6 x 3), b2: 1 x C (1 x 3)
W2 = [[random.gauss(0, 0.1) for _ in range(C)] for _ in range(H)]
b2 = [[0.0 for _ in range(C)]]

lr = 0.5
epochs = 100

print("=== 開始純 Python 零庫訓練 ===")

# ----------------------------------------------------
# 3. 訓練主循環
# ----------------------------------------------------
for epoch in range(epochs):
    # ================================================
    # A. 前向傳播 (Forward)
    # ================================================
    # 1) Z1 = X @ W1 + b1  -> (N x H)
    Z1 = add_bias(matmul(X, W1), b1)
    
    # 2) A1 = ReLU(Z1)     -> (N x H)
    A1 = relu_forward(Z1)
    
    # 3) Z2 = A1 @ W2 + b2 -> (N x C)
    Z2 = add_bias(matmul(A1, W2), b2)
    
    # 4) A2, Loss = Softmax_CE(Z2, y)
    A2, loss = softmax_cross_entropy_forward(Z2, y)
    
    # ================================================
    # B. 反向傳播 (Backward - 鏈式法則)
    # ================================================
    # 1) 求 dZ2 -> (N x C)
    dZ2 = softmax_cross_entropy_backward(A2, y)
    
    # 2) dW2 = A1^T @ dZ2 -> (H x C)
    #    db2 = sum(dZ2, axis=0) -> (1 x C)
    dW2 = matmul(transpose(A1), dZ2)
    db2 = [[sum(dZ2[i][j] for i in range(N)) for j in range(C)]]
    
    # 3) dA1 = dZ2 @ W2^T -> (N x H)
    dA1 = matmul(dZ2, transpose(W2))
    
    # 4) 穿透 ReLU 得到 dZ1 -> (N x H)
    dZ1 = relu_backward(dA1, Z1)
    
    # 5) dW1 = X^T @ dZ1 -> (D x H)
    #    db1 = sum(dZ1, axis=0) -> (1 x H)
    dW1 = matmul(transpose(X), dZ1)
    db1 = [[sum(dZ1[i][j] for i in range(N)) for j in range(H)]]
    
    # ================================================
    # C. 參數更新 (Gradient Descent)
    # ================================================
    # W1 = W1 - lr * dW1
    W1 = mat_sub(W1, mat_scale(dW1, lr))
    b1 = mat_sub(b1, mat_scale(db1, lr))
    W2 = mat_sub(W2, mat_scale(dW2, lr))
    b2 = mat_sub(b2, mat_scale(db2, lr))
    
    # 每 20 輪計算一次正確率
    if (epoch + 1) % 20 == 0 or epoch == 0:
        correct_count = 0
        for i in range(N):
            # 找出最大概率對應的類別索引 argmax
            pred_class = A2[i].index(max(A2[i]))
            if pred_class == y[i]:
                correct_count += 1
        acc = (correct_count / N) * 100
        print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f} | Accuracy: {acc:.2f}%")


=== 開始純 Python 零庫訓練 ===
Epoch   1 | Loss: 1.1007 | Accuracy: 34.00%
Epoch  20 | Loss: 1.0904 | Accuracy: 37.00%
Epoch  40 | Loss: 1.0782 | Accuracy: 39.00%
Epoch  60 | Loss: 1.0592 | Accuracy: 43.00%
Epoch  80 | Loss: 1.0486 | Accuracy: 44.00%
Epoch 100 | Loss: 1.0438 | Accuracy: 46.00%
